# Gold — fact_monthly_performance

`silver.monthly_performance` + `gold.dim_ticker` → **`gold.fact_monthly_performance`**.

**Grain: one row per (ticker, month).** This is the raw material — what happened, month by
month. The horizon fact rolls it up into the answer.

The one thing this notebook really does is resolve the **versioned** `ticker_key`: each month
is joined to the `dim_ticker` version that was live *in that month*. Done once here, so every
later query is a plain equi-join with no date range to forget.

Expected: **15,604 rows**.

In [0]:
CREATE TABLE IF NOT EXISTS `index-vs-trust-pipeline`.gold.fact_monthly_performance (
  monthly_key  STRING  COMMENT 'MD5(ticker|month_key)',
  ticker_key   STRING  COMMENT 'The dim_ticker version live in this month, resolved at build time',
  month_key    INT     COMMENT 'Joins dim_date',
  ticker       STRING  COMMENT 'Readable, so the fact can be eyeballed without a join',
  close        DOUBLE,
  dividend     DOUBLE,
  price_return DOUBLE  COMMENT 'Null across a gap, never zero',
  total_return DOUBLE  COMMENT 'Null for the archive trusts: the CSV carries no dividends',
  return_basis STRING  COMMENT 'total, or price for the two archive trusts'
)
COMMENT 'Monthly return for every trust and index, at the grain (ticker, month)';

In [0]:
CREATE OR REPLACE TEMP VIEW gold_stage_fact_monthly AS
WITH priced AS (
  SELECT ticker, month_key, close, dividend, price_return, total_return, return_basis
  FROM `index-vs-trust-pipeline`.silver.monthly_performance
),
versioned AS (
  -- The whole reason the surrogate key exists: each month attaches to the version of the
  -- trust that was live that month, so "returns while X managed it" is later a plain join.
  SELECT p.*, d.ticker_key
  FROM priced p
  JOIN `index-vs-trust-pipeline`.gold.dim_ticker d
    ON d.ticker = p.ticker
   AND p.month_key BETWEEN d.effective_start_month
                       AND COALESCE(d.effective_end_month, 999912)
)
SELECT MD5(CONCAT_WS('|', ticker, CAST(month_key AS STRING))) AS monthly_key,
       ticker_key, month_key, ticker, close, dividend,
       price_return, total_return, return_basis
FROM versioned;

In [0]:
MERGE INTO `index-vs-trust-pipeline`.gold.fact_monthly_performance AS t
USING gold_stage_fact_monthly AS s
   ON t.ticker = s.ticker AND t.month_key = s.month_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

## Verification

In [0]:
SELECT COUNT(*)                                                      AS rows_total,
       COUNT(*) - COUNT(DISTINCT monthly_key)                        AS duplicate_keys,
       COUNT(DISTINCT ticker)                                        AS tickers,
       SUM(CASE WHEN return_basis = 'price' THEN 1 ELSE 0 END)       AS price_basis_rows,
       (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.silver.monthly_performance) AS silver_rows
FROM `index-vs-trust-pipeline`.gold.fact_monthly_performance;

Expect **15,604 / 0 / 96 / 270 / 15,604**.

`rows_total` and `silver_rows` must match: the fact loses nothing, it only adds keys. The 270
price-basis rows are exactly `BCPT` and `CSH`, which have no dividends to build a total
return from.

In [0]:
-- Referential integrity: every key in the fact must exist in the dimensions.
SELECT SUM(CASE WHEN d.ticker_key IS NULL THEN 1 ELSE 0 END) AS orphan_ticker_keys,
       SUM(CASE WHEN dt.month_key IS NULL THEN 1 ELSE 0 END) AS orphan_month_keys
FROM `index-vs-trust-pipeline`.gold.fact_monthly_performance f
LEFT JOIN `index-vs-trust-pipeline`.gold.dim_ticker d ON d.ticker_key = f.ticker_key
LEFT JOIN `index-vs-trust-pipeline`.gold.dim_date   dt ON dt.month_key = f.month_key;

Expect **0 / 0**. A fact row pointing at a key that does not exist is a broken star.